In [ ]:
import streamlit as st
from transformers import WhisperForConditionalGeneration, WhisperProcessor, pipeline
import huggingface_hub
import io
from pydub import AudioSegment
from pydub.playback import play
from audiorecorder import audiorecorder


In [ ]:

# Authenticate with your user token
hf_token = st.secrets["HF_TOKEN"]
huggingface_hub.login(token=hf_token)


In [ ]:

def load_model():
    # Load the model and processor
    model = WhisperForConditionalGeneration.from_pretrained("CiBeDL/twi_trained_whisper")
    processor = WhisperProcessor.from_pretrained("CiBeDL/twi_trained_whisper")

    # Initialize the pipeline
    asr_pipeline = pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,  # Use the tokenizer from the processor
        feature_extractor=processor.feature_extractor,
    )
    return asr_pipeline


In [ ]:

# Load the ASR pipeline
pipe = load_model()

# Streamlit app interface
st.title("Asanti-Twi Speech Transcription")
st.write("Upload an audio file and get the transcription.")

# File uploader
audio_file = st.file_uploader("Upload audio file", type=["wav", "mp3", "ogg", "opus"])

if audio_file is not None:
    # Play the audio file
    st.audio(audio_file, format=f"audio/{audio_file.name.split('.')[-1]}")

    # Perform inference
    with st.spinner("Transcribing..."):
        try:
            # Read audio data and transcribe
            audio_data = io.BytesIO(audio_file.read())  # Ensure the audio file is read as bytes
            transcription = pipe(audio_data.read())["text"]
        except Exception as e:
            st.error(f"Error during transcription: {e}")
            st.stop()  # Stop execution if an error occurs

    st.success("Transcription complete!")
    st.write(transcription)
